# Notebook 03 — Covariance Ellipses and Geometry

## Learning objectives
- Link eigendecomposition to ellipse shape and orientation.
- Compare from-scratch ellipse math with `kiss_slam.math_utils.covariance_ellipse_params`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from kiss_slam.math_utils import covariance_ellipse_params

np.random.seed(19)


## 1) Ellipse geometry from covariance

For $\Sigma = Q\Lambda Q^T$, major/minor axis lengths at $n$-sigma are:

$$
2n\sqrt{\lambda_1},\quad 2n\sqrt{\lambda_2}.
$$

The major axis orientation is the first eigenvector direction.


In [ ]:
def from_scratch(mean, cov, n_std=2.0):
    vals, vecs = np.linalg.eigh(cov)
    order = np.argsort(vals)[::-1]
    vals, vecs = vals[order], vecs[:,order]
    width, height = 2*n_std*np.sqrt(np.clip(vals, 0, None))
    angle = np.degrees(np.arctan2(vecs[1,0], vecs[0,0]))
    return np.asarray(mean), float(width), float(height), float(angle)

mean = np.array([2.0, -1.0])
cov = np.array([[3.2, 1.1],[1.1, 1.2]])
print('scratch:', from_scratch(mean, cov))
print('repo   :', covariance_ellipse_params(mean, cov))


## 2) Compare implementations visually


In [ ]:
mine = from_scratch(mean, cov)
repo = covariance_ellipse_params(mean, cov)
L = np.linalg.cholesky(cov)
pts = (mean.reshape(2,1) + L @ np.random.normal(size=(2,2000))).T

fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(pts[:,0], pts[:,1], s=4, alpha=0.2)
ax.add_patch(Ellipse(mine[0], mine[1], mine[2], angle=mine[3], fill=False, lw=3, color='C3', label='scratch'))
ax.add_patch(Ellipse(repo[0], repo[1], repo[2], angle=repo[3], fill=False, lw=1.5, ls='--', color='k', label='repo'))
ax.set_aspect('equal'); ax.legend(); plt.show()


## 3) Interactive sweep (edit and rerun)


In [ ]:
mean = np.array([0.0, 0.0])
var_x, var_y, cov_xy = 5.0, 2.0, -2.0
cov = np.array([[var_x, cov_xy],[cov_xy, var_y]])
print(cov)
print('det=', np.linalg.det(cov))
if np.linalg.det(cov) > 0:
    c, w, h, a = covariance_ellipse_params(mean, cov, n_std=2.0)
    print('angle=', a)
    L = np.linalg.cholesky(cov)
    pts = (L @ np.random.normal(size=(2,1500))).T
    fig, ax = plt.subplots(figsize=(6,6))
    ax.scatter(pts[:,0], pts[:,1], s=4, alpha=0.2)
    ax.add_patch(Ellipse(c, w, h, angle=a, fill=False, lw=2, color='C1'))
    ax.set_aspect('equal'); plt.show()
else:
    print('Non-PD covariance; reduce |cov_xy|.')


## Exercises

1. Predict ellipse rotation sign from covariance by inspection.
2. Compute axis lengths for eigenvalues 9 and 1 at 2-sigma.

## Common mistakes
- Not sorting eigenvalues before assigning major/minor axis.
- Confusing full axis length with semi-axis length.
- Ignoring 180° angle equivalence.
